[Reference](https://medium.com/versent-tech-blog/training-an-llm-with-hugging-face-363942f1a237)

```
pip install transformers
pip install torch
pip install datasets
pip install 'transformers[torch]'
pip install peft
```

# Starting Point

In [1]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")
question  = "When was ACDC formed?"
knowledge = """
    ACDC is the name of a band that was formed in Sydney in 1973.
    The members of the band include Malcolm as the rhythm guitarist and Angus as the lead guitarist.
"""
result = qa("Context: " + knowledge + " Question: " + question)

print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'generated_text': '1973'}]


# Fine-Tuning

## Training Data

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import Trainer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

In [3]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("json", data_files="acdc_qa.json")

def preprocess(example):
    inputs = tokenizer(example["question"], max_length=128, truncation=False, padding="max_length")
    targets = tokenizer(example["answer"], max_length=128, truncation=False, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, peft_config)

training_args = Seq2SeqTrainingArguments(
    output_dir="./acdc-finetuned-model",
    per_device_train_batch_size=8,
    num_train_epochs=100,
    logging_steps=1,
    push_to_hub=False,
    learning_rate=1e-3,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['train'].select(range(20)),
)

trainer.train()

model.save_pretrained("./acdc-finetuned-model")
tokenizer.save_pretrained("./acdc-finetuned-model")

# Using the Fine Tuned Model

In [4]:
from transformers import pipeline

qa_pipeline = pipeline("text2text-generation", model="./acdc-finetuned-model", tokenizer="./acdc-finetuned-model")

questions = [
    "When was ACDC formed?",
    "Where was ACDC formed?",
    "List the members of Cold Chisel.",
    "List the members of ACDC.",
]

for question in questions:
    answer = qa_pipeline(question)
    print(f"{question} Answer: {answer[0]['generated_text']}")